In [42]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.impute import SimpleImputer
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

In [43]:
# Loading the fao dataset to be used for predicting yield for different crops
fao_final_df = pd.read_csv('faostat_model_dataset_clean.csv')
fao_final_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 252 entries, 0 to 251
Data columns (total 34 columns):
 #   Column                                  Non-Null Count  Dtype  
---  ------                                  --------------  -----  
 0   area                                    252 non-null    object 
 1   item                                    252 non-null    object 
 2   year                                    252 non-null    int64  
 3   domestic_supply_quantity                252 non-null    float64
 4   export_quantity                         251 non-null    float64
 5   feed                                    113 non-null    float64
 6   food                                    252 non-null    float64
 7   import_quantity                         244 non-null    float64
 8   losses                                  252 non-null    float64
 9   other_uses_non_food                     76 non-null     float64
 10  processing                              117 non-null    float6

In [44]:
yield_df_final = pd.read_csv('yield_cleaned.csv')
yield_df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4396 entries, 0 to 4395
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   item             4396 non-null   object 
 1   year             4396 non-null   int64  
 2   unit             4396 non-null   object 
 3   yield_kg_per_ha  4396 non-null   float64
dtypes: float64(1), int64(1), object(2)
memory usage: 137.5+ KB


In [45]:
fao_final_df = fao_final_df.merge(
    yield_df_final,
    on=['item', 'year'],
    how='left'
)

print("Final dataset shape:", fao_final_df.shape)

print("\nMissing values (top 15):")
print(fao_final_df.isna().sum().sort_values(ascending=False).head(15))

print("\nSample:")
print(fao_final_df.head())

Final dataset shape: (252, 36)

Missing values (top 15):
yield_kg_per_ha             224
unit                        224
other_uses_non_food         176
feed                        139
processing                  135
seed                        131
stock_variation              32
import_quantity               8
export_quantity               1
year                          0
item                          0
area                          0
domestic_supply_quantity      0
losses                        0
production                    0
dtype: int64

Sample:
    area     item  year  domestic_supply_quantity  export_quantity  feed  \
0  Kenya  Bananas  2010                    1583.0              0.0   NaN   
1  Kenya  Bananas  2011                    1227.0              0.0   NaN   
2  Kenya  Bananas  2012                    1208.0              0.0   NaN   
3  Kenya  Bananas  2013                    1375.0              0.0   NaN   
4  Kenya  Bananas  2014                    1645.0            

Creating lags for our data

- Lag features(yield lags and production lags) will help capture trends over time, seasonality of crops and also momentum effect

In [46]:
fao_final_df = fao_final_df.sort_values(['area', 'item', 'year']).copy()

In [47]:
# Yield lags
fao_final_df['yield_lag_1'] = fao_final_df.groupby(['area', 'item'])['yield_kg_per_ha'].shift(1)
fao_final_df['yield_lag_2'] = fao_final_df.groupby(['area', 'item'])['yield_kg_per_ha'].shift(2)

# Production lags
fao_final_df['production_lag_1'] = fao_final_df.groupby(['area', 'item'])['production'].shift(1)
fao_final_df['production_lag_2'] = fao_final_df.groupby(['area', 'item'])['production'].shift(2)

In [48]:
print(fao_final_df[
    ['yield_lag_1','yield_lag_2','production_lag_1','production_lag_2']
].isnull().sum())

yield_lag_1         226
yield_lag_2         228
production_lag_1     18
production_lag_2     36
dtype: int64


In [49]:
fao_final_df['yield_kg_per_ha'].unique()

array([18968.4, 24287.3, 20823.8, 22850.3, 27705.6, 21248.2, 20357.2,
       18487.1, 19439.3, 22308.3, 25742.8, 27695.8, 28587.7, 24212.6,
           nan,  9973.9, 12268.9, 12834.6, 12470.6, 12505. , 17077.3,
       14779.7, 12097.7, 13545.9, 16975.6, 12696.3, 11818.3, 12239.5,
       12294.4])

In [50]:
fao_final_df.describe()

,year,domestic_supply_quantity,export_quantity,feed,food,import_quantity,losses,other_uses_non_food,processing,production,...,gdp_usd_2015,agriculture_value_added_usd_2015,real_price_lcu_per_tonne,gdp_kes_2015_equiv,agriculture_value_added_kes_2015_equiv,yield_kg_per_ha,yield_lag_1,yield_lag_2,production_lag_1,production_lag_2
count,252.000000,252.000000,251.000000,113.000000,252.000000,244.000000,252.000000,76.000000,117.000000,252.000000,...,252.000000,252.000000,252.000000,2.520000e+02,2.520000e+02,28.000000,26.000000,24.000000,234.000000,216.000000
mean,2016.500000,796.166667,20.290837,99.132743,685.936508,217.352459,41.011905,23.618421,22.931624,605.035714,...,75643.196763,13942.124225,59829.463319,7.790066e+06,1.419150e+06,18081.860714,18068.657692,17873.245833,594.145299,593.462963
std,4.039151,1133.913818,41.328139,226.362679,945.378621,510.512087,54.386069,35.517495,23.962516,931.777487,...,13405.618608,1339.672689,93395.407582,2.562197e+06,3.459431e+05,5745.699130,5728.062085,5416.246059,909.996338,915.394488
min,2010.000000,-43.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,11.000000,...,55768.208320,11821.199977,7323.577526,4.418691e+06,9.366309e+05,9973.900000,9973.900000,9973.900000,11.000000,11.000000
25%,2013.000000,71.750000,0.000000,0.000000,54.500000,1.000000,4.000000,2.000000,0.000000,72.750000,...,63571.883415,12900.631019,25383.011881,5.474994e+06,1.111039e+06,12496.400000,12552.825000,12648.475000,72.250000,73.000000
50%,2016.500000,180.000000,2.000000,9.000000,146.500000,7.000000,15.000000,4.000000,18.000000,162.000000,...,74494.823721,13752.632144,35653.903577,7.633859e+06,1.408971e+06,17782.200000,17782.200000,17782.200000,159.000000,154.500000
75%,2020.000000,1090.250000,18.000000,40.000000,1072.500000,38.250000,53.500000,42.000000,39.000000,710.500000,...,84294.124217,15239.429327,53033.731122,8.948700e+06,1.652212e+06,22443.800000,22043.275000,21513.225000,711.500000,716.500000
max,2023.000000,5221.000000,227.000000,1247.000000,4035.000000,2557.000000,245.000000,171.000000,84.000000,4285.000000,...,100109.927037,16224.299115,848699.462957,1.400001e+07,2.268910e+06,28587.700000,28587.700000,27705.600000,4014.000000,4014.000000


## YIELD PREDICTION

## Model 1: Random Forest Regresser Model

In predicting the yield, we used the following key features:

- fertilizers(N,P,K) -This directly affect the growth of any crop

- pesticides - basically crop protection chemicals that help reduce losses from pests, weeds and disease

- fertilzers and pesticiders are the strongest biological drivers of yield.

- Economic indicators(GDP,CPI,exchange rate)

- supply variables(production, imports,domestic suply quantity import quality)- will assist with market avalability and also food system pressure.

In [51]:

# Features
features = [
    'nitrogen', 'phosphorus', 'potassium',
    'pesticides_total', 'fungicides', 'herbicides', 'insecticides',
    'gdp_kes_2015_equiv', 'agriculture_value_added_kes_2015_equiv',
    'food_cpi', 'kes_per_usd',
    'production', 'domestic_supply_quantity', 'import_quantity',
    'price_lcu_per_tonne', 'yield_lag_1','yield_lag_2','production_lag_1','production_lag_2',
]

df_model = fao_final_df.dropna(subset=['yield_kg_per_ha']).copy()

# splitting our data using time test split
train = df_model[df_model['year'] <= 2020]
test = df_model[df_model['year'] > 2020]


# Features & target
X_train = train[features]
y_train = train['yield_kg_per_ha']

X_test = test[features]
y_test = test['yield_kg_per_ha']


# handling the missing values from the latest merging
imputer = SimpleImputer(strategy="median")

X_train = imputer.fit_transform(X_train)
X_test = imputer.transform(X_test)

# Model
model_yield = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

# fitting the model
model_yield.fit(X_train, y_train)

# predicting our model
y_pred = model_yield.predict(X_test)


print("Random Forest Yield Prediction (Time Split)")
print("MAE:", mean_absolute_error(y_test, y_pred))
print("R² :", r2_score(y_test, y_pred))

Random Forest Yield Prediction (Time Split)
MAE: 2097.360500000015
R² : 0.8640598109608766


# Random Forest Regressor Model result interpratation
From our statistics, the **mean yield_per_kg_ha is 18,081.86kgs**.

- From the MAE, our model **predicition is off by 2097.36kgs with an error of 11.59%**.

- The R² : 0.8640598109608766, indicates that the model can explain 86% of the variation variation in yield.


## Model 2: XGBoost Model

In [52]:
features = [
    'nitrogen', 'phosphorus', 'potassium',
    'pesticides_total', 'fungicides', 'herbicides', 'insecticides',
    'gdp_kes_2015_equiv', 'agriculture_value_added_kes_2015_equiv',
    'food_cpi', 'kes_per_usd',
    'production', 'domestic_supply_quantity', 'import_quantity',
    'price_lcu_per_tonne','yield_lag_1','yield_lag_2', 'production_lag_1','production_lag_2'
]

df_model = fao_final_df.dropna(subset=['yield_kg_per_ha']).copy()

# For our time series data, we will use a time-split
train = df_model[df_model['year'] <= 2020]
test = df_model[df_model['year'] > 2020]

# Defining our target and features
X_train2 = train[features]
y_train2 = train['yield_kg_per_ha']

X_test2 = test[features]
y_test2 = test['yield_kg_per_ha']

# Handdling the missing values that arose from merging the previous dataset with the yield dataset
imputer = SimpleImputer(strategy="median")

X_train2 = imputer.fit_transform(X_train)
X_test2 = imputer.transform(X_test)

# XGB
model_xgb = XGBRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

# fiting the model
model_xgb.fit(X_train2, y_train2)

# Evaluation of the model
y_pred2 = model_xgb.predict(X_test2)



mae = mean_absolute_error(y_test2, y_pred2)
rsme = np.sqrt(mean_squared_error(y_test2, y_pred2))
r2 = r2_score(y_test2, y_pred2)

print("XGBoost Yield Prediction Model (Time Split)")
print(f"MAE: {mae:.2f}")
print(f"R² : {r2:.2f}")
print(f"RMSE: {rsme:.2f}")


XGBoost Yield Prediction Model (Time Split)
MAE: 2367.84
R² : 0.81
RMSE: 3242.75


### XGBoost Yield Prediction Model Intepretation

From our statistics, **the mean yield_per_kg_ha is 18081.86**. Our models predicitions are **off by 2368kgs** from our MAE metric.

- The R² : 0.8120046957187079, indicated that the model can explain 81% of yield variation.
Our model has an error of 13.09%.

- This means **our model can explain 81.2% of the variation in yield**
- We did not use **Adjusted R² because yield is not particularly linear**

## PRICE PREDICTION

In [53]:
kamis_df = pd.read_csv('kamis_data.csv',index_col=False)
kamis_df.drop(columns=['Unnamed: 0'], inplace=True)
kamis_df.head()

,Commodity,Market,Wholesale,Retail,Supply Volume,County,Date,commodity,commodity_id
0,Dry Maize,Ngurubani Market,50.0,60.00,5800.0,Kirinyaga,2026-04-17,Dry Maize,1
1,Dry Maize,Kerugoya,55.0,80.00,4600.0,Kirinyaga,2026-04-17,Dry Maize,1
2,Dry Maize,Voi Retail,0.0,65.00,360.0,Taita-Taveta,2026-04-17,Dry Maize,1
3,Dry Maize,Kawangware,55.0,65.00,0.0,Nairobi,2026-04-17,Dry Maize,1
4,Dry Maize,Mumias,0.0,61.36,0.0,Kakamega,2026-04-17,Dry Maize,1


In [54]:
kamis_df.columns

Index(['Commodity', 'Market', 'Wholesale', 'Retail', 'Supply Volume', 'County',
       'Date', 'commodity', 'commodity_id'],
      dtype='object')

In [55]:
kamis_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 432628 entries, 0 to 432627
Data columns (total 9 columns):
 #   Column         Non-Null Count   Dtype  
---  ------         --------------   -----  
 0   Commodity      432628 non-null  object 
 1   Market         432628 non-null  object 
 2   Wholesale      432628 non-null  float64
 3   Retail         432628 non-null  float64
 4   Supply Volume  432628 non-null  float64
 5   County         432628 non-null  object 
 6   Date           432628 non-null  object 
 7   commodity      432628 non-null  object 
 8   commodity_id   432628 non-null  int64  
dtypes: float64(3), int64(1), object(5)
memory usage: 29.7+ MB


In [56]:
# fix column names
kamis_df.columns = kamis_df.columns.str.lower().str.replace(" ", "_")

# convert date
kamis_df['date'] = pd.to_datetime(kamis_df['date'])

# extract time features
kamis_df['year'] = kamis_df['date'].dt.year
kamis_df['month'] = kamis_df['date'].dt.month

print(kamis_df.head())

   commodity            market  wholesale  retail  supply_volume  \
0  Dry Maize  Ngurubani Market       50.0   60.00         5800.0   
1  Dry Maize          Kerugoya       55.0   80.00         4600.0   
2  Dry Maize        Voi Retail        0.0   65.00          360.0   
3  Dry Maize        Kawangware       55.0   65.00            0.0   
4  Dry Maize            Mumias        0.0   61.36            0.0   

         county       date  commodity  commodity_id  year  month  
0     Kirinyaga 2026-04-17  Dry Maize             1  2026      4  
1     Kirinyaga 2026-04-17  Dry Maize             1  2026      4  
2  Taita-Taveta 2026-04-17  Dry Maize             1  2026      4  
3       Nairobi 2026-04-17  Dry Maize             1  2026      4  
4      Kakamega 2026-04-17  Dry Maize             1  2026      4  


In [57]:
kamis_df.columns = kamis_df.columns.str.lower()

In [58]:
# standardise column names
kamis_df.columns = kamis_df.columns.str.lower().str.replace(" ", "_")

# remove duplicate columns
kamis_df = kamis_df.loc[:, ~kamis_df.columns.duplicated()]

print(kamis_df.columns.tolist())

['commodity', 'market', 'wholesale', 'retail', 'supply_volume', 'county', 'date', 'commodity_id', 'year', 'month']


In [59]:
kamis_df = kamis_df.sort_values(['commodity', 'date'])

### Creating lags

In [ ]:
kamis_df = kamis_df.sort_values(['commodity', 'date'])


# PRICE LAGS

kamis_df['retail_lag_1'] = kamis_df.groupby('commodity')['retail'].shift(1)
kamis_df['retail_lag_7'] = kamis_df.groupby('commodity')['retail'].shift(7)

kamis_df['wholesale_lag_1'] = kamis_df.groupby('commodity')['wholesale'].shift(1)
kamis_df['wholesale_lag_7'] = kamis_df.groupby('commodity')['wholesale'].shift(7)

# ROLLING FEATURES
kamis_df['retail_rolling_7'] = (
    kamis_df.groupby('commodity')['retail']
      .rolling(7)
      .mean()
      .reset_index(0, drop=True)
)

kamis_df['wholesale_rolling_7'] = (
    kamis_df.groupby('commodity')['wholesale']
      .rolling(7)
      .mean()
      .reset_index(0, drop=True)
)

# PRICE SPREAD
kamis_df['price_spread'] = kamis_df['retail'] - kamis_df['wholesale']

# VOLATILITY
kamis_df['price_volatility_7'] = (
    kamis_df.groupby('commodity')['retail']
      .rolling(7)
      .std()
      .reset_index(0, drop=True)
)

# PRICE CHANGE
kamis_df['price_change_1'] = kamis_df.groupby('commodity')['retail'].diff(1)


# SUPPLY FEATURES
kamis_df['supply_lag_1'] = kamis_df.groupby('commodity')['supply_volume'].shift(1)

kamis_df['supply_rolling_7'] = (
    kamis_df.groupby('commodity')['supply_volume']
      .rolling(7)
      .mean()
      .reset_index(0, drop=True)
)

kamis_df['supply_change_1'] = kamis_df.groupby('commodity')['supply_volume'].diff(1)

# RELATIVE PRICE LEVEL
kamis_df['price_vs_avg_7'] = kamis_df['retail'] / kamis_df['retail_rolling_7']


# CLEAN
kamis_df = kamis_df.dropna()

print(kamis_df.shape)
print(kamis_df.head())

(402367, 23)
                     commodity market  wholesale  retail  supply_volume  \
101512  African butter catfish   Aram        0.0   250.0           50.0   
101511  African butter catfish   Aram        0.0   250.0           32.0   
101510  African butter catfish   Aram        0.0   250.0           40.0   
101509  African butter catfish   Aram        0.0   250.0           40.0   
101508  African butter catfish   Aram        0.0   250.0           38.5   

       county       date  commodity_id  year  month  ...  wholesale_lag_7  \
101512  Siaya 2021-08-02            87  2021      8  ...              0.0   
101511  Siaya 2021-08-05            87  2021      8  ...            180.0   
101510  Siaya 2021-08-09            87  2021      8  ...              0.0   
101509  Siaya 2021-08-12            87  2021      8  ...              0.0   
101508  Siaya 2021-08-16            87  2021      8  ...              0.0   

        retail_rolling_7  wholesale_rolling_7  price_spread  \
101512    

## Creating features

In [61]:
features = [
    'retail_lag_1',
    'retail_lag_7',
    'wholesale_lag_1',
    'wholesale_lag_7',
    'retail_rolling_7',
    'wholesale_rolling_7',
    'price_spread',
    'price_volatility_7',
    'price_change_1',
    'supply_volume',
    'supply_lag_1',
    'supply_rolling_7',
    'supply_change_1',
    'price_vs_avg_7',
    'month',
    'commodity_id'
]

X = kamis_df[features]
y = kamis_df['retail']

print(X.shape)
print(X.head())


(402367, 16)
        retail_lag_1  retail_lag_7  wholesale_lag_1  wholesale_lag_7  \
101512         250.0         250.0              0.0              0.0   
101511         250.0         250.0              0.0            180.0   
101510         250.0         250.0              0.0              0.0   
101509         250.0         250.0              0.0              0.0   
101508         250.0         250.0              0.0              0.0   

        retail_rolling_7  wholesale_rolling_7  price_spread  \
101512             250.0            25.714286         250.0   
101511             250.0             0.000000         250.0   
101510             250.0             0.000000         250.0   
101509             250.0             0.000000         250.0   
101508             250.0             0.000000         250.0   

        price_volatility_7  price_change_1  supply_volume  supply_lag_1  \
101512                 0.0             0.0           50.0          32.0   
101511                 0.

## Train_test_split

In [62]:
train = kamis_df[kamis_df['date'] <= '2023-12-31']
test = kamis_df[kamis_df['date'] > '2023-12-31']

X_train = train[features]
y_train = train['retail']

X_test = test[features]
y_test = test['retail']

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

Train size: (228711, 16)
Test size: (173656, 16)


## Modelling

In [63]:
from xgboost import XGBRegressor

model = XGBRegressor(
    n_estimators=100,
    max_depth=6,        # slightly deeper for big data
    learning_rate=0.1,
    random_state=42,
    n_jobs=-1           # use all cores
)


#fitting the model
model.fit(X_train, y_train)

#predicting the model
y_pred = model.predict(X_test)


mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)



print("Test RMSE:", rmse)
print("MAE:",  mae)




Test RMSE: 780.8872853892668
MAE: 9.694910677696143


For XGBoost, R2 AND adjusted R2 are not very meaningful because:

they assumes linear relationships and XGBoost is non-linear

In [64]:
errors = abs(y_test - y_pred)

worst_idx = errors.sort_values(ascending=False).head(10).index

kamis_df.loc[worst_idx, ['commodity', 'date', 'retail']]

,commodity,date,retail
215998,Dry Onions,2024-04-26,9000.0
4429,Dry Maize,2024-02-13,3600.0
6040,Red Sorghum,2024-10-09,7000.0
344938,Fertilizer,2024-02-13,6000.0
405973,Banana (Plantain),2025-10-10,45000.0
6826,Red Sorghum,2024-05-22,7500.0
498,Dry Maize,2026-01-16,5000.0
420051,Lentils,2024-10-09,15500.0
63419,Beans Rosecoco,2024-10-09,16200.0
324099,Pigeon peas,2024-10-09,17100.0


1. Train separate models per commodity

In [65]:
id="fix_per_commodity"

all_preds = []
all_actuals = []

for commodity in kamis_df['commodity'].unique():
    train_c = train[train['commodity'] == commodity]
    test_c = test[test['commodity'] == commodity]

    if len(test_c) < 10:
        continue

    X_train_c = train_c[features]
    y_train_c = train_c['retail']

    X_test_c = test_c[features]
    y_test_c = test_c['retail']

    model = XGBRegressor(
        n_estimators=400,
        max_depth=4,
        learning_rate=0.03,
        random_state=42
    )


    model.fit(X_train_c, y_train_c)
    preds = model.predict(X_test_c)

    all_preds.extend(preds)
    all_actuals.extend(y_test_c)

# Evaluate globally
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

rmse = np.sqrt(mean_squared_error(all_actuals, all_preds))
mae = mean_absolute_error(all_actuals, all_preds)

print("Fixed RMSE:", rmse)
print("Fixed MAE:", mae)

Fixed RMSE: 209.69140822754323
Fixed MAE: 11.987320741045574
